# UMAP-DEA Results Explorer

**How to use:** Edit the configuration in the cell below, then run all cells (Cell → Run All).
Change any parameter values and re-run to see updated plots.

In [ ]:
# ============================================================
# CONFIGURATION — change these values to filter the results
# ============================================================

# Which N (number of inputs) to include. Options: 20, 50, 100, 200
SELECTED_N = [100]

# Which n (number of DMUs) to include. Options: 20, 50, 100, 200
SELECTED_n = [20, 50, 100, 200]

# Which returns-to-scale to include. Options: 'vrs', 'crs', or 'both'
SELECTED_RTS = 'crs'

# Whether to include runs with PCA or not. Options: True, False, 'both'
SELECTED_PCA = False

# Which metric to plot. Options:
#   'MAE', 'Spearman', 'Pearson', 'Kendall',
#   'Proportion Efficient', 'Number Efficient'
METRIC = 'Kendall'

# Show ±1 standard deviation as error bars? (True / False)
SHOW_STD = True

# ============================================================

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)
%matplotlib inline

In [ ]:
# ============================================================
# 1. LOAD ALL DATA
# ============================================================

# Resolve path relative to this notebook's location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
RESULTS_DIR = os.path.join(NOTEBOOK_DIR, '..', 'results')
RESULTS_DIR = os.path.abspath(RESULTS_DIR)
print(f'Looking for results in: {RESULTS_DIR}')

# Load all summary_df files
summary_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'summary_df_*.csv')))
params_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'params_dict_*.csv')))
print(f'Found {len(summary_files)} summary files and {len(params_files)} params files.')

if len(summary_files) == 0:
    raise FileNotFoundError(f'No summary_df files found in {RESULTS_DIR}. Check the path.')

# Load all params (one row per run_id)
params_list = []
for f in params_files:
    df = pd.read_csv(f)
    run_id = os.path.basename(f).replace('params_dict_', '').replace('.csv', '')
    df['run_id'] = run_id
    params_list.append(df)
params_all = pd.concat(params_list, ignore_index=True)
print(f'Loaded {len(params_all)} parameter configurations.')

# Load all summaries (5 rows per run_id = one per dim_reduction_level)
summary_list = []
for f in summary_files:
    df = pd.read_csv(f)
    run_id = os.path.basename(f).replace('summary_df_', '').replace('.csv', '')
    df['run_id'] = run_id
    summary_list.append(df)
summary_all = pd.concat(summary_list, ignore_index=True)
print(f'Loaded {len(summary_all)} summary rows.')

# Merge
data = summary_all.merge(params_all, on='run_id', how='left')
print(f'Merged: {data.shape[0]} rows × {data.shape[1]} columns.\n')

In [ ]:
# ============================================================
# 2. FILTER DATA
# ============================================================

df = data.copy()

# Apply N filter
if SELECTED_N:
    df = df[df['N'].isin(SELECTED_N)]

# Apply n filter
if SELECTED_n:
    df = df[df['n'].isin(SELECTED_n)]

# Apply RTS filter
if SELECTED_RTS != 'both':
    df = df[df['rts'] == SELECTED_RTS]

# Apply PCA filter
if SELECTED_PCA != 'both':
    df = df[df['pca'] == SELECTED_PCA]

print(f'After filtering: {len(df)} rows from {df["run_id"].nunique()} unique runs.')
print(f'  N values present:   {sorted(df["N"].unique())}')
print(f'  n values present:   {sorted(df["n"].unique())}')
print(f'  RTS values present: {sorted(df["rts"].unique())}')
print(f'  PCA values present: {sorted(df["pca"].unique())}')
print(f'  Dim reduction levels: {sorted(df["dim_reduction_level"].unique())}')

if len(df) == 0:
    raise ValueError('No data matches the current filter. Broaden selection.')

In [ ]:
# ============================================================
# 3. MAP METRIC NAME TO COLUMNS
# ============================================================

METRIC_MAP = {
    'MAE':                  ('mae_mean', 'mae_std'),
    'Spearman':             ('spearmanr_mean', 'spearmanr_std'),
    'Pearson':              ('pearsonr_mean', 'pearsonr_std'),
    'Kendall':              ('kendalltau_mean', 'kendalltau_std'),
    'Proportion Efficient': ('prop_efficient_mean', 'prop_efficient_std'),
    'Number Efficient':     ('nr_efficient_mean', 'nr_efficient_std'),
}

if METRIC not in METRIC_MAP:
    raise ValueError(f'Unknown metric "{METRIC}". Choose from: {list(METRIC_MAP.keys())}')

mean_col, std_col = METRIC_MAP[METRIC]
print(f'Plotting: {METRIC}  (columns: {mean_col}, {std_col})')

In [ ]:
# ============================================================
# 4. PLOT — Panel A: Metric vs Dim Reduction Level
# ============================================================

DIM_ORDER = ['log', 'sqrt', 'ten_percent', 'half', 'original']
DIM_LABELS = ['log(N)', '√N', '10%', 'N/2', 'N (original)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Panel A: Metric vs dim_reduction_level ──
# Group by (N, n, rts, pca, dim_reduction_level) and average across run_ids
group_cols = ['N', 'n', 'rts', 'pca', 'dim_reduction_level']
agg = df.groupby(group_cols, dropna=False).agg(
    metric_avg=(mean_col, 'mean'),
    metric_std=(std_col, 'mean'),
).reset_index()

# Create a single label for each (N, n, rts, pca) combination
agg['label'] = agg.apply(
    lambda r: f'N={int(r["N"])}, n={int(r["n"])}, {r["rts"].upper()}, {"PCA" if r["pca"] else "NoPCA"}', axis=1
)

# Ensure ordering
agg['dim_order'] = pd.Categorical(agg['dim_reduction_level'], categories=DIM_ORDER, ordered=True)
agg = agg.sort_values(['label', 'dim_order'])

# Plot one line per label
labels_unique = sorted(agg['label'].unique())
colors = plt.cm.tab20(np.linspace(0, 1, len(labels_unique)))

for i, label in enumerate(labels_unique):
    subset = agg[agg['label'] == label]
    x_pos = np.arange(len(subset))
    y_vals = subset['metric_avg'].values
    y_errs = subset['metric_std'].values if SHOW_STD else None
    
    # Skip if all NaN
    if np.all(np.isnan(y_vals)):
        continue
    
    ax1.errorbar(
        x_pos, y_vals, yerr=y_errs,
        marker='o', markersize=6, linewidth=2, capsize=4,
        color=colors[i], label=label,
    )

ax1.set_xticks(range(len(DIM_ORDER)))
ax1.set_xticklabels(DIM_LABELS, rotation=30, ha='right')
ax1.set_xlabel('Dimension Reduction Level')
ax1.set_ylabel(METRIC)
ax1.set_title(f'{METRIC} by Dimension Reduction Level')
ax1.legend(fontsize=7, loc='best', ncol=1)
ax1.grid(True, alpha=0.3)

# ── Panel B: Warning counts ──
warn_agg = df.groupby('dim_reduction_level', dropna=False).agg(
    spearman_warns=('spearmanr_warning_count', 'mean'),
    kendall_warns=('kendalltau_warning_count', 'mean'),
).reindex(DIM_ORDER).reset_index()

x = np.arange(len(warn_agg))
width = 0.35
bars1 = ax2.bar(x - width/2, warn_agg['spearman_warns'], width,
                label='Spearman Warnings', color='#E57373', edgecolor='white')
bars2 = ax2.bar(x + width/2, warn_agg['kendall_warns'], width,
                label='Kendall Warnings', color='#64B5F6', edgecolor='white')

ax2.set_xticks(x)
ax2.set_xticklabels(DIM_LABELS, rotation=30, ha='right')
ax2.set_ylabel('Mean Warning Count')
ax2.set_title('DEA Warning Counts by Dim Reduction Level')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

# Annotate bars
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., h + max(warn_agg['spearman_warns'].max(), warn_agg['kendall_warns'].max())*0.02,
                f'{int(h)}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., h + max(warn_agg['spearman_warns'].max(), warn_agg['kendall_warns'].max())*0.02,
                f'{int(h)}', ha='center', va='bottom', fontsize=8)

# Title
fig.suptitle(
    f'Results Explorer — {METRIC}  |  '
    f'N ∈ {sorted(df["N"].unique())}, n ∈ {sorted(df["n"].unique())}, '
    f'RTS={"both" if SELECTED_RTS == "both" else SELECTED_RTS}, '
    f'PCA={"both" if SELECTED_PCA == "both" else SELECTED_PCA}  |  '
    f'{df["run_id"].nunique()} run(s)',
    fontsize=13, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. DATA TABLE
# ============================================================

# Aggregate: one row per (dim_reduction_level, N, n, rts, pca)
table_agg = df.groupby(group_cols, dropna=False).agg(
    metric_avg=(mean_col, 'mean'),
    metric_std=(std_col, 'mean'),
    n_runs=('run_id', 'nunique'),
).reset_index()

# Rename columns for display
table_agg = table_agg.rename(columns={
    'dim_reduction_level': 'DimReduction',
    'metric_avg': f'{METRIC} (mean)',
    'metric_std': f'{METRIC} (std)',
    'n_runs': '#Runs',
})
table_agg['pca'] = table_agg['pca'].map({True: 'PCA', False: 'NoPCA'})
table_agg['rts'] = table_agg['rts'].str.upper()

# Sort for readability
table_agg = table_agg.sort_values(['N', 'n', 'rts', 'pca', 'DimReduction'])

# Display with color gradient
display_cols = ['DimReduction', 'N', 'n', 'rts', 'pca', f'{METRIC} (mean)', f'{METRIC} (std)', '#Runs']
styled = table_agg[display_cols].style \
    .background_gradient(subset=[f'{METRIC} (mean)'], cmap='RdYlGn_r') \
    .format({f'{METRIC} (mean)': '{:.5f}', f'{METRIC} (std)': '{:.5f}'})
display(styled)

print(f'\nTable: {len(table_agg)} rows.')

---
### Parameter Reference

| Parameter | Values in dataset |
|---|---|
| N (inputs) | 20, 50, 100, 200 |
| n (DMUs) | 20, 50, 100, 200 |
| RTS | vrs, crs |
| PCA | True, False |
| dim_reduction_level | log, sqrt, ten_percent, half, original |

### Metrics Available

| Metric | Description |
|---|---|
| MAE | Mean Absolute Error |
| Spearman | Spearman rank correlation |
| Pearson | Pearson correlation |
| Kendall | Kendall tau correlation |
| Proportion Efficient | Proportion of DMUs classified as efficient |
| Number Efficient | Number of DMUs classified as efficient |